In [ ]:
from google.colab import drive
import subprocess

drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/Colab Notebooks/Gestion_Datos_IA_EV2'

subprocess.run([
    'jupyter', 'nbconvert', '--to', 'notebook',
    '--execute', f'{CARPETA}/Creacion_de_datos_EV2.ipynb',
    '--output', '/content/Creacion_de_datos_EV2_ejecutado.ipynb'
], check=True)

print("Creación de datos completada ✓")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creación de datos completada ✓


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd
import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
log = logging.getLogger("pipeline_fraude")

CARPETA = '/content/drive/MyDrive/Colab Notebooks/Gestion_Datos_IA_EV2'
df_raw = pd.read_csv(f'{CARPETA}/datos_generados.csv')
log.info(f"Dataset cargado desde Drive: {df_raw.shape}")

df = df_raw.copy()

# --- 2.1 Revisar nulos ---
nulos = df.isnull().sum()
log.info(f"Valores nulos por columna:\n{nulos[nulos > 0]}")

# --- 2.2 Duplicados ---
duplicados = df.duplicated().sum()
log.info(f"Filas duplicadas: {duplicados}")
df = df.drop_duplicates()

# --- 2.3 Crear features nuevas ---
df['hour']         = pd.to_datetime(df['trans_date_trans_time']).dt.hour
df['is_night']     = df['hour'].apply(lambda h: 1 if h >= 22 or h <= 5 else 0)
df['geo_distance'] = np.sqrt(
    (df['lat'] - df['merch_lat'])**2 +
    (df['long'] - df['merch_long'])**2
) * 111
df['amt_zscore']   = df.groupby('category')['amt'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

# --- 2.4 Codificar categóricas ---
le_gender   = LabelEncoder()
le_category = LabelEncoder()
df['gender_enc']   = le_gender.fit_transform(df['gender'])
df['category_enc'] = le_category.fit_transform(df['category'])
log.info(f"Categorías codificadas: {dict(zip(le_category.classes_, le_category.transform(le_category.classes_)))}")

# --- 2.5 Seleccionar features ---
FEATURES = [
    'amt',
    'hour',          # ← corregido (era 'hora')
    'is_night',      # ← corregido (era 'es_noche')
    'geo_distance',  # ← corregido (era 'diferencia_geo')
    'amt_zscore',
    'city_pop',
    'category_enc',
    'gender_enc',
    'lat',
    'merch_lat',
    'merch_long'
]
TARGET = 'is_fraud'

df_clean = df[FEATURES + [TARGET]].copy()

log.info(f"Shape después de limpieza: {df_clean.shape}")
log.info(f"Features seleccionadas: {FEATURES}")

# --- 2.6 Guardar para el siguiente notebook ---
df_clean.to_csv(f'{CARPETA}/datos_limpios.csv', index=False)
log.info("datos_limpios.csv guardado en Drive")

pd.set_option('display.float_format', '{:.2f}'.format)
print(df_clean.describe())

           amt     hour  is_night  geo_distance  amt_zscore   city_pop  \
count 19000.00 19000.00  19000.00      19000.00    19000.00   19000.00   
mean    292.37    11.13      0.37        157.18       -0.00 1533482.78   
std     664.25     7.10      0.48        562.92        1.00  858289.67   
min       5.01     0.00      0.00          0.62       -1.73   50517.00   
25%      83.02     5.00      0.00         32.03       -0.44  781182.25   
50%     160.50    11.00      0.00         45.51       -0.27 1538471.00   
75%     237.74    17.00      1.00         55.76        0.46 2280827.25   
max    4999.37    23.00      1.00       5753.47        4.84 2999953.00   

       category_enc  gender_enc      lat  merch_lat  merch_long  is_fraud  
count      19000.00    19000.00 19000.00   19000.00    19000.00  19000.00  
mean           3.11        0.50    36.46      36.47      -95.00      0.05  
std            2.00        0.50     6.61       6.62       14.50      0.22  
min            0.00        0.